# Election Dataset Assembly

This notebook builds the working election dataset by combining selected variables from multiple source files.

Goals:
- Identify required variables
- Load and clean source datasets
- Merge them into one analysis-ready table
- Validate the final dataset
- Save outputs for reuse in the project and GitHub repository

**Import Libraries**

In [24]:
from pathlib import Path
import pandas as pd
import os

**Define Working Directories**

In [19]:
project_dir = Path.cwd()
data_dir = project_dir / "data"
raw_dir = data_dir / "raw"
processed_dir = data_dir / "processed"
output_dir = project_dir / "output"

## Project Setup
The notebook starts by defining the working directories that will be used to load source files and save outputs.

In [20]:
raw_dir, processed_dir, output_dir

(WindowsPath('C:/Users/mutch/Democratic_Voter_Shift/data/raw/data/raw'),
 WindowsPath('C:/Users/mutch/Democratic_Voter_Shift/data/raw/data/processed'),
 WindowsPath('C:/Users/mutch/Democratic_Voter_Shift/data/raw/output'))

## Source Data Inventory

The following is a list of each dataset and the variables taken from them.

1)  Democratic_Voter_Shift/data/raw/a_countypres_2000-2024.csv
       1a) 
3)  Democratic_Voter_Shift/data/raw/a_sources-president.csv
4)  Democratic_Voter_Shift/data/raw/d_pa_cvap_2023_cnty.csv
5)  Democratic_Voter_Shift/data/raw/e_currentvotestats.xlsx
6)  Democratic_Voter_Shift/data/raw/a_County Presidential Returns 2000-2024.md
7)  


**REDUCING DATASET #1 to meaningful variables: a_countypres_2000-2024.cvs**

*Step 1a: Keep only PA rows*

check current working directory

In [25]:
print("Current working directory:", os.getcwd())

Current working directory: C:\Users\mutch\Democratic_Voter_Shift\data\raw


check to see if file exists at the specified path

In [32]:
file_path = 'a_countypres_2000-2024.csv'
if os.path.exists(file_path):
    print("File found!")
    df_pa = pd.read_csv(file_path)
else:
    print("File not found at specified path.")

File found!


In [33]:
# Load the raw CSV file
df_pa = pd.read_csv('a_countypres_2000-2024.csv')

In [36]:
# Keep only Pennsylvania rows
df_pa = df_pa[df_pa["state"].astype(str).str.strip().str.upper() == "PENNSYLVANIA"].copy()

# Preview the result
df_pa.head()

,state,county_name,year,state_po,county_fips,office,candidate,party,candidatevotes,totalvotes,version,mode
13710,PENNSYLVANIA,ADAMS,2024,PA,42001.0,US PRESIDENT,CHASE OLIVER,LIBERTARIAN,238.0,60650,20260225,ELECTION DAY
13711,PENNSYLVANIA,ADAMS,2024,PA,42001.0,US PRESIDENT,DONALD J TRUMP,REPUBLICAN,30642.0,60650,20260225,ELECTION DAY
13712,PENNSYLVANIA,ADAMS,2024,PA,42001.0,US PRESIDENT,KAMALA D HARRIS,DEMOCRAT,10843.0,60650,20260225,ELECTION DAY
13713,PENNSYLVANIA,ADAMS,2024,PA,42001.0,US PRESIDENT,OTHER,OTHER,140.0,60650,20260225,ELECTION DAY
13714,PENNSYLVANIA,ADAMS,2024,PA,42001.0,US PRESIDENT,CHASE OLIVER,LIBERTARIAN,102.0,60650,20260225,MAIL-IN


*Step 1b: Drop columns 'state', 'office' & 'version'*

In [37]:
df_pa = df_pa.drop(columns=["state", "office", "version"])

*Step 1c: Save the altered file*

In [48]:
df_pa.to_csv("C:/Users/mutch/Democratic_Voter_Shift/data/processed/a_countypres_2000-2024_pennsylvania_only.csv", index=False)

In [40]:
df_pa.head()

,county_name,year,state_po,county_fips,candidate,party,candidatevotes,totalvotes,mode
13710,ADAMS,2024,PA,42001.0,CHASE OLIVER,LIBERTARIAN,238.0,60650,ELECTION DAY
13711,ADAMS,2024,PA,42001.0,DONALD J TRUMP,REPUBLICAN,30642.0,60650,ELECTION DAY
13712,ADAMS,2024,PA,42001.0,KAMALA D HARRIS,DEMOCRAT,10843.0,60650,ELECTION DAY
13713,ADAMS,2024,PA,42001.0,OTHER,OTHER,140.0,60650,ELECTION DAY
13714,ADAMS,2024,PA,42001.0,CHASE OLIVER,LIBERTARIAN,102.0,60650,MAIL-IN


In [57]:
# Load the Pennsylvania-only dataset
df = pd.read_csv("C:/Users/mutch/Democratic_Voter_Shift/data/processed/a_countypres_2000-2024_pennsylvania_only.csv")

# Sum candidatevotes by county, candidate, and mode
# Keep the other columns by taking the first value within each group
df_grouped = (
    df.groupby(["county_name", "candidate", "year"], as_index=False)
      .agg({
          "candidatevotes": "sum",
          "state_po": "first",
          "county_fips": "first",
          "party": "first"
      })
)

df_grouped.to_csv(
    "C:/Users/mutch/Democratic_Voter_Shift/data/processed/a_countypres_2000-2024_pa_county_candidate_mode_summed.csv",
    index=False
)

df_grouped.head()

,county_name,candidate,year,candidatevotes,state_po,county_fips,party
0,ADAMS,AL GORE,2000,11682.0,PA,42001.0,DEMOCRAT
1,ADAMS,BARACK OBAMA,2008,17633.0,PA,42001.0,DEMOCRAT
2,ADAMS,BARACK OBAMA,2012,15091.0,PA,42001.0,DEMOCRAT
3,ADAMS,CHASE OLIVER,2024,684.0,PA,42001.0,LIBERTARIAN
4,ADAMS,DONALD J TRUMP,2020,37523.0,PA,42001.0,REPUBLICAN


Checking names for consistency

In [58]:
candidate_list = (
    df["candidate"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

candidate_list = sorted(candidate_list)

print(candidate_list)

['AL GORE', 'BARACK OBAMA', 'CHASE OLIVER', 'DONALD J TRUMP', 'DONALD TRUMP', 'GEORGE W. BUSH', 'HILLARY CLINTON', 'JO JORGENSEN', 'JOHN KERRY', 'JOHN MCCAIN', 'JOSEPH R BIDEN JR', 'KAMALA D HARRIS', 'MITT ROMNEY', 'OTHER', 'RALPH NADER']


In [60]:
df_pa.to_csv("C:/Users/mutch/Democratic_Voter_Shift/data/processed/a_countypres_2000-2024_testGrouped.csv", index=False)

In [62]:
df = pd.read_csv("C:/Users/mutch/Democratic_Voter_Shift/data/processed/a_countypres_2000-2024_testGrouped.csv")

df = df[
    df["year"].isin([2020, 2024]) &
    (df["mode"].astype(str).str.strip().str.upper() == "TOTAL")
].copy()

df.to_csv(
    "C:/Users/mutch/Democratic_Voter_Shift/data/processed/a_countypres_2020_2024_total_only.csv",
    index=False
)

df.head()




,county_name,year,state_po,county_fips,candidate,party,candidatevotes,totalvotes,mode
12,ADAMS,2024,PA,42001.0,CHASE OLIVER,LIBERTARIAN,342.0,60650,TOTAL
13,ADAMS,2024,PA,42001.0,DONALD J TRUMP,REPUBLICAN,40248.0,60650,TOTAL
14,ADAMS,2024,PA,42001.0,KAMALA D HARRIS,DEMOCRAT,19842.0,60650,TOTAL
15,ADAMS,2024,PA,42001.0,OTHER,OTHER,218.0,60650,TOTAL
28,ALLEGHENY,2024,PA,42003.0,CHASE OLIVER,LIBERTARIAN,3302.0,720504,TOTAL


As all rows are vote totals, the 'mode' column will be dropped

In [64]:
df = df.drop(columns=["mode"])

In [65]:
df.head()

,county_name,year,state_po,county_fips,candidate,party,candidatevotes,totalvotes
12,ADAMS,2024,PA,42001.0,CHASE OLIVER,LIBERTARIAN,342.0,60650
13,ADAMS,2024,PA,42001.0,DONALD J TRUMP,REPUBLICAN,40248.0,60650
14,ADAMS,2024,PA,42001.0,KAMALA D HARRIS,DEMOCRAT,19842.0,60650
15,ADAMS,2024,PA,42001.0,OTHER,OTHER,218.0,60650
28,ALLEGHENY,2024,PA,42003.0,CHASE OLIVER,LIBERTARIAN,3302.0,720504


Saving 8 variable file

In [66]:
df.to_csv("C:/Users/mutch/Democratic_Voter_Shift/data/processed/a_countypres_2020-2024_total_only.csv", index=False)

**THE FOLLOWING CELL** creates a WIDE format using the candidate name.  In hindsight, this creates a 
problem with missing values between years becuase candidates for the parties change.  Following this transformation,
another will be made using 'party' rather than candidate.

In [67]:
df_wide = (
    df.pivot_table(
        index=["county_name", "year", "state_po", "county_fips", "totalvotes"],
        columns="party",
        values="candidatevotes",
        aggfunc="sum"
    )
    .reset_index()
)
df_wide.head(20)

candidate,county_name,year,state_po,county_fips,totalvotes,CHASE OLIVER,DONALD J TRUMP,JO JORGENSEN,JOSEPH R BIDEN JR,KAMALA D HARRIS,OTHER
0,ADAMS,2020,PA,42001.0,56540,NaN,37523.0,810.0,18207.0,NaN,NaN
1,ADAMS,2024,PA,42001.0,60650,342.0,40248.0,NaN,NaN,19842.0,218.0
2,ALLEGHENY,2020,PA,42003.0,719733,NaN,282324.0,8344.0,429065.0,NaN,NaN
3,ALLEGHENY,2024,PA,42003.0,720504,3302.0,283595.0,NaN,NaN,429916.0,3691.0
4,ARMSTRONG,2020,PA,42005.0,36370,NaN,27489.0,424.0,8457.0,NaN,NaN
5,ARMSTRONG,2024,PA,42005.0,37119,182.0,28296.0,NaN,NaN,8553.0,88.0
6,BEAVER,2020,PA,42007.0,94122,NaN,54759.0,1241.0,38122.0,NaN,NaN
7,BEAVER,2024,PA,42007.0,94843,502.0,56837.0,NaN,NaN,37196.0,308.0
8,BEDFORD,2020,PA,42009.0,27574,NaN,23025.0,182.0,4367.0,NaN,NaN
9,BEDFORD,2024,PA,42009.0,28160,100.0,23658.0,NaN,NaN,4336.0,66.0


**THE FOLLOWING** cells delete the 'candidate name', and create a WIDE format using 'party'

In [68]:
df = pd.read_csv("C:/Users/mutch/Democratic_Voter_Shift/data/processed/a_countypres_2020-2024_total_only.csv")

In [69]:
df = df.drop(columns=["candidate"])
df_wideP = (
    df.pivot_table(
        index=["county_name", "year", "state_po", "county_fips", "totalvotes"],
        columns="party",
        values="candidatevotes",
        aggfunc="sum"
    )
    .reset_index()
)
df_wideP.head(20)

party,county_name,year,state_po,county_fips,totalvotes,DEMOCRAT,LIBERTARIAN,OTHER,REPUBLICAN
0,ADAMS,2020,PA,42001.0,56540,18207.0,810.0,NaN,37523.0
1,ADAMS,2024,PA,42001.0,60650,19842.0,342.0,218.0,40248.0
2,ALLEGHENY,2020,PA,42003.0,719733,429065.0,8344.0,NaN,282324.0
3,ALLEGHENY,2024,PA,42003.0,720504,429916.0,3302.0,3691.0,283595.0
4,ARMSTRONG,2020,PA,42005.0,36370,8457.0,424.0,NaN,27489.0
5,ARMSTRONG,2024,PA,42005.0,37119,8553.0,182.0,88.0,28296.0
6,BEAVER,2020,PA,42007.0,94122,38122.0,1241.0,NaN,54759.0
7,BEAVER,2024,PA,42007.0,94843,37196.0,502.0,308.0,56837.0
8,BEDFORD,2020,PA,42009.0,27574,4367.0,182.0,NaN,23025.0
9,BEDFORD,2024,PA,42009.0,28160,4336.0,100.0,66.0,23658.0


In [71]:
df_wideP.to_csv("C:/Users/mutch/Democratic_Voter_Shift/data/processed/a_countypres_2020-2024_partytotals.csv", index=False)

In [72]:
df_wideP.head(30)

party,county_name,year,state_po,county_fips,totalvotes,DEMOCRAT,LIBERTARIAN,OTHER,REPUBLICAN
0,ADAMS,2020,PA,42001.0,56540,18207.0,810.0,NaN,37523.0
1,ADAMS,2024,PA,42001.0,60650,19842.0,342.0,218.0,40248.0
2,ALLEGHENY,2020,PA,42003.0,719733,429065.0,8344.0,NaN,282324.0
3,ALLEGHENY,2024,PA,42003.0,720504,429916.0,3302.0,3691.0,283595.0
4,ARMSTRONG,2020,PA,42005.0,36370,8457.0,424.0,NaN,27489.0
5,ARMSTRONG,2024,PA,42005.0,37119,8553.0,182.0,88.0,28296.0
6,BEAVER,2020,PA,42007.0,94122,38122.0,1241.0,NaN,54759.0
7,BEAVER,2024,PA,42007.0,94843,37196.0,502.0,308.0,56837.0
8,BEDFORD,2020,PA,42009.0,27574,4367.0,182.0,NaN,23025.0
9,BEDFORD,2024,PA,42009.0,28160,4336.0,100.0,66.0,23658.0


### SUMMARY
This will be the base datafile to which the other files are added.
